# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DMACN - Deep Multi-kernel Auto-encoder Clustering Network
First load the data

In [ ]:
from scipy.io import loadmat
import torch

# Load the data
PTSD = loadmat("C:\\Users\\oddar\\Downloads\\PTSD_connectivity.mat")
# PTSD is a dataset containing 87 samples (subjects) with 340 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 3

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = PTSD["connectivities"]  # Example matrix

PTSD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = PTSD_tensor  # [N,d] float tensor

epoch    1/500 [mid-only]  J=2.0591e+03  J1=1.8945e+03  J2=5.1654e-06  J3=1.6454e+02  omega_sum=1.0000
epoch   50/500 [multilayer]  J=6.2845e+02  J1=4.6785e+02  J2=1.0961e-05  J3=1.6060e+02  omega_sum=1.0000
epoch  100/500 [multilayer]  J=6.0401e+02  J1=4.5046e+02  J2=4.4868e-06  J3=1.5356e+02  omega_sum=1.0000
epoch  150/500 [multilayer]  J=3.3744e+02  J1=1.8868e+02  J2=2.5425e-04  J3=1.4876e+02  omega_sum=1.0000
epoch  200/500 [multilayer]  J=3.2993e+02  J1=1.8785e+02  J2=2.0426e-04  J3=1.4208e+02  omega_sum=1.0000
epoch  250/500 [multilayer]  J=3.1901e+02  J1=1.8339e+02  J2=1.9670e-04  J3=1.3563e+02  omega_sum=1.0000
epoch  300/500 [multilayer]  J=3.1244e+02  J1=1.8295e+02  J2=1.9257e-04  J3=1.2949e+02  omega_sum=1.0000
epoch  350/500 [multilayer]  J=3.0911e+02  J1=1.8548e+02  J2=1.9009e-04  J3=1.2363e+02  omega_sum=1.0000
epoch  400/500 [multilayer]  J=2.9992e+02  J1=1.8183e+02  J2=2.0119e-04  J3=1.1809e+02  omega_sum=1.0000
epoch  450/500 [multilayer]  J=2.9342e+02  J1=1.8062e+02 

Define the autoencoder specs

In [ ]:
from DMACN import DMACN, DMACNConfig

kernel_specs = [
    {"kind": "rbf", "t": 0.01},
    {"kind": "rbf", "t": 0.05},
    {"kind": "rbf", "t": 0.1},
    {"kind": "rbf", "t": 1},
    {"kind": "rbf", "t": 10},
    {"kind": "rbf", "t": 50},
    {"kind": "rbf", "t": 100},
    {"kind": "poly", "a": 0, "b": 2},
    {"kind": "poly", "a": 0, "b": 4},
    {"kind": "poly", "a": 1, "b": 2},
    {"kind": "poly", "a": 1, "b": 4}
]  # h = 3

cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[340, 285, 240, 202, 170],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[170, 202, 240, 285, 340],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

Run the model

In [ ]:
model = DMACN(cfg)
model.fit(X, verbose_every=50)
labels = model.predict(save=True)
print("labels shape:", labels.shape)
print("labels: ", labels)

### UMAP

In [ ]:
from UMAP import UMAP

### Evaluate the clusters
Evaluate the clusters using simple methods: Silhouette coefficient, Davies-Bouldin score and Calinski-Harabasz score

In [ ]:
from Evaluate_models import evaluate_clustering
import os

# Define where to find the labels 
labels_path = os.fsencode("Clusters")
evaluate_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, labels_path=labels_path)